# Vector stores and semantic search

En este notebook se implementa un vector store desde cero para realizar busquedas semanticas con embeddings y similitud coseno.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

## Part I: Basic vector store implementation

In [2]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        if not documents:
            return

        texts = [document.text for document in documents]
        new_embeddings = self.embedding_model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        self.documents.extend(documents)
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if not self.documents or self.embeddings is None:
            return []

        query_embedding = self.embedding_model.encode(
            query,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        scores = self.embeddings @ query_embedding
        result_count = min(top_k, len(self.documents))
        top_indices = np.argsort(scores)[::-1][:result_count]

        return [
            SearchResult(score=float(scores[index]), document=self.documents[index])
            for index in top_indices
        ]

## Animal Fun Facts Dataset

In [3]:
animal_data_path = Path("data/animal-fun-facts-dataset.csv")
animal_df = pd.read_csv(animal_data_path).fillna("")

animal_documents = [
    Document(
        text=str(row["text"]),
        metadata={
            "animal_name": str(row["animal_name"]),
            "source": str(row["source"]),
            "media_link": str(row["media_link"]),
            "wikipedia_link": str(row["wikipedia_link"]),
        },
    )
    for _, row in animal_df.iterrows()
]


In [4]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


def print_search_results(results: list[SearchResult], max_text_length: int = 350):
    for position, result in enumerate(results, start=1):
        text = result.document.text.replace("\n", " ")
        if len(text) > max_text_length:
            text = text[:max_text_length].rstrip() + "..."

        print("Resultado", position)
        print("Score:", format(result.score, ".4f"))
        print("Texto:", text)
        print("Metadatos:", result.document.metadata)
        print("-" * 80)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
animal_vector_store = VectorStore(embedding_model)
animal_vector_store.add_documents(animal_documents)

## Example semantic searches with VectorStore

In [6]:
animal_queries = [
    "animals that eat ants and termites",
    "large mammals that live in the ocean",
    "birds that cannot fly",
    "animals with strong shells or armor",
    "fast predators from Africa",
]

for query in animal_queries:
    print("Consulta:", query)
    print_search_results(animal_vector_store.search(query, top_k=3))

Consulta: animals that eat ants and termites
Resultado 1
Score: 0.7827
Texto: What they eat can depend on where they are located. Ants and termites aren’t always found in the same places. As a result, while the giant anteater is known to eat both, they might not always.
Metadatos: {'animal_name': 'giant anteater', 'source': 'https://factanimal.com/giant-anteater/', 'media_link': '', 'wikipedia_link': '/wiki/Giant_anteater'}
--------------------------------------------------------------------------------
Resultado 2
Score: 0.7131
Texto: They can consume 30,000 insects a day. They visit up to 200 termite or ants nests a day and spend on average around a minute feeding, before moving on to the next nest.
Metadatos: {'animal_name': 'giant anteater', 'source': 'https://factanimal.com/giant-anteater/', 'media_link': '', 'wikipedia_link': '/wiki/Giant_anteater'}
--------------------------------------------------------------------------------
Resultado 3
Score: 0.7106
Texto: Giant anteaters pr

## Part II: Filtering by metadata

In [7]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        if not documents:
            return

        texts = [document.text for document in documents]
        new_embeddings = self.embedding_model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        self.documents.extend(documents)
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        if not self.documents or self.embeddings is None:
            return []

        metadata_filter = metadata_filter or {}
        candidate_indices = [
            index
            for index, document in enumerate(self.documents)
            if all(document.metadata.get(key) == value for key, value in metadata_filter.items())
        ]
        if not candidate_indices:
            return []

        query_embedding = self.embedding_model.encode(
            query,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        candidate_embeddings = self.embeddings[candidate_indices]
        scores = candidate_embeddings @ query_embedding
        result_count = min(top_k, len(candidate_indices))
        top_positions = np.argsort(scores)[::-1][:result_count]

        return [
            SearchResult(
                score=float(scores[position]),
                document=self.documents[candidate_indices[position]],
            )
            for position in top_positions
        ]

## IMDb review dataset with metadata

In [8]:
def load_imdb_documents(base_path: Path, samples_per_sentiment: int = 500) -> list[Document]:
    documents = []
    sentiment_folders = {
        "positive": base_path / "train" / "pos",
        "negative": base_path / "train" / "neg",
    }

    for sentiment, folder in sentiment_folders.items():
        files = sorted(folder.glob("*.txt"))[:samples_per_sentiment]
        for file_path in files:
            _, rating = file_path.stem.split("_")
            documents.append(
                Document(
                    text=file_path.read_text(encoding="utf-8"),
                    metadata={
                        "sentiment": sentiment,
                        "rating": rating,
                    },
                )
            )

    return documents


imdb_base_path = Path("data/aclImdb_v1/aclImdb")
imdb_documents = load_imdb_documents(imdb_base_path, samples_per_sentiment=500)
imdb_vector_store = FilteredVectorStore(embedding_model)
imdb_vector_store.add_documents(imdb_documents)

## Example semantic searches with FilteredVectorStore

In [9]:
filtered_queries = [
    ("excellent acting and memorable characters", {"sentiment": "positive"}),
    ("boring story with weak writing", {"sentiment": "negative"}),
    ("a wonderful movie with a strong emotional ending", {"sentiment": "positive", "rating": "10"}),
    ("terrible movie that wastes time", {"sentiment": "negative", "rating": "1"}),
    ("funny comedy with clever jokes", {"sentiment": "positive"}),
]

for query, metadata_filter in filtered_queries:
    print("Consulta:", query)
    print("Filtro:", metadata_filter)
    results = imdb_vector_store.search(query, top_k=3, metadata_filter=metadata_filter)
    print_search_results(results)

Consulta: excellent acting and memorable characters
Filtro: {'sentiment': 'positive'}
Resultado 1
Score: 0.5933
Texto: The makers have chosen the best people for the job, and set the scene wonderfully. Every interior is full of detail that tells you all about the people who live in it. Whether the period is the 20s (the first story), the present (ie 1950) for the middle story, or the 1910s (the last), costumes and settings are lovingly observed and created. I love...
Metadatos: {'sentiment': 'positive', 'rating': '8'}
--------------------------------------------------------------------------------
Resultado 2
Score: 0.5541
Texto: Meryl Streep was incredible in this film. She has an amazing knack for accents, and she shows incredible skill in this film overall. I really felt for her when Lindy was being persecuted. She was played realistically, too. She got cranky, upset, and unpleasant as the media and the government continued their unrelenting witchhunt. I didn't expect mu...
Metadato

## Reflexiones personales

En esta actividad entendi como funciona un vector store desde cero. La idea principal es convertir cada documento en un embedding y despues comparar la consulta con esos embeddings usando similitud coseno. Tambien vi que los resultados dependen mucho del contenido del dataset y de que tan relacionada este la consulta con los textos disponibles, porque algunas busquedas funcionaron bien y otras no fueron tan precisas.

La parte de FilteredVectorStore me parecio util porque permite combinar busqueda semantica con filtros por metadatos, como sentimiento o rating. Esto hace que los resultados sean mas controlados y mas cercanos a una aplicacion real. 